**1.connect to duckdb**

In [1]:
import duckdb
import pandas as pd
import plotly.express as px

con = duckdb.connect()


**2.create view with merged train sets**

In [2]:
#count the number of rows in each parquet file
separate_counts = con.execute("""
    SELECT 
        filename, 
        count(*) as row_count
    FROM read_parquet('Data/train-part*/*.parquet', filename=True)
    GROUP BY filename
    ORDER BY filename
""").df()
display(separate_counts)

#create a view that combines all the parquet files and count the total number of rows across all files
con.execute("""
    CREATE OR REPLACE VIEW all_train AS 
    SELECT * EXCLUDE(session_end_completed), CAST(session_end_completed AS INT) AS session_end_completed
    FROM read_parquet('Data/train-part*/*.parquet')
""")

total_count = con.execute("""
    SELECT 
        count(*) as total_rows
    FROM all_train
""").fetchone()[0]
print(f"Total number of rows across all training files: {total_count}")

#show the first 5 rows of the combined view
sample_rows = con.execute("""
    SELECT * 
    FROM all_train
    LIMIT 5
""").df()
display(sample_rows)

#show the content of the 'history' column for the first row
print(sample_rows['history'].iloc[0])

#take a sample of 100000 rows from all_train
sample_data = con.execute("""
    SELECT * 
    FROM all_train
    USING SAMPLE 100000 ROWS
""").df()


,filename,row_count
0,Data\train-part\part-00000-9b4bba6b-feac-44b1-...,25613243
1,Data\train-part\part-00001-9b4bba6b-feac-44b1-...,25501113
2,Data\train-part\part-00002-9b4bba6b-feac-44b1-...,36551483


Total number of rows across all training files: 87665839


,datetime,ui_language,eligible_templates,history,selected_template,session_end_completed
0,0.153461,en,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'A', 'n_days': 28.19564819335937...",B,0
1,2.827303,es,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'A', 'n_days': 29.836181640625},...",A,1
2,2.792662,en,"[G, E, B, K, H, J, L, F, D]","[{'template': 'G', 'n_days': 8.197543144226074...",J,1
3,4.904225,en,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'B', 'n_days': 29.00238037109375...",L,0
4,9.538715,en,"[K, H, G, E, B, J, L, F, D, A]","[{'template': 'B', 'n_days': 27.97145843505859...",B,1


[{'template': 'A', 'n_days': 28.195648193359375}
 {'template': 'C', 'n_days': 27.19352912902832}
 {'template': 'C', 'n_days': 26.134174346923828}
 {'template': 'B', 'n_days': 25.127830505371094}
 {'template': 'C', 'n_days': 23.19371223449707}
 {'template': 'C', 'n_days': 21.19255256652832}
 {'template': 'E', 'n_days': 20.146591186523438}
 {'template': 'A', 'n_days': 19.130592346191406}
 {'template': 'G', 'n_days': 18.126190185546875}
 {'template': 'B', 'n_days': 17.133644104003906}
 {'template': 'E', 'n_days': 15.943687438964844}
 {'template': 'C', 'n_days': 14.28361988067627}
 {'template': 'B', 'n_days': 13.193897247314453}
 {'template': 'A', 'n_days': 12.135659217834473}
 {'template': 'G', 'n_days': 11.134573936462402}
 {'template': 'E', 'n_days': 10.134580612182617}
 {'template': 'A', 'n_days': 9.142278671264648}
 {'template': 'F', 'n_days': 8.142279624938965}
 {'template': 'H', 'n_days': 7.113645076751709}
 {'template': 'E', 'n_days': 6.024662494659424}
 {'template': 'A', 'n_days':

**3.explore the features**

In [3]:
#calculate the success rate of all train observations
success_rate = con.execute("""
    SELECT
        AVG(session_end_completed) AS success_rate
    FROM sample_data
""").fetchone()[0]

print(f"success rate: {success_rate:.2%}")

#calculate the success rate by 'ui_language'
ui_language_success_rates = con.execute("""
    SELECT
        ui_language,
        AVG(session_end_completed) AS success_rate
    FROM sample_data
    GROUP BY ui_language
    ORDER BY success_rate DESC
""").df()

fig_language = px.bar(
    ui_language_success_rates, 
    x='ui_language', 
    y='success_rate',
    hover_data={'success_rate': ':.2%'},
    text_auto='.2%', 
    title='Success Rate by UI Language',
    labels={'success_rate': 'Success Rate', 'ui_language': 'Language'},
    color='success_rate',
    color_continuous_scale='Viridis'
)
fig_language.show()

#calculate the success rate by 'selected template'
selected_template_success_rates = con.execute("""
    SELECT
        selected_template,
        AVG(session_end_completed) AS success_rate
    FROM sample_data
    GROUP BY selected_template
    ORDER BY success_rate DESC
""").df()

fig_template = px.bar(
    selected_template_success_rates, 
    x='selected_template', 
    y='success_rate',
    hover_data={'success_rate': ':.2%'},
    text_auto='.2%', 
    title='Success Rate by Selected Template',
    labels={'success_rate': 'Success Rate', 'selected_template': 'Template'},
    color='success_rate',
    color_continuous_scale='Viridis'
)
fig_template.show()


success rate: 14.30%


In [4]:
#count the number of rows in sample_data where 'history' is null or empty
null_history_count = con.execute(""" 
SELECT COUNT(*) AS null_history_count
FROM sample_data 
WHERE history IS NULL OR len(history) = 0
""").fetchone()[0]
print(f"Number of rows with null or empty history: {null_history_count}")

Number of rows with null or empty history: 7301


In [5]:
#extract from 'history' column the number of days since the selected template was last sent and calculate the success rate by this variable
con.execute("""
    CREATE OR REPLACE VIEW history_extracted AS 
    SELECT 
    *,
    COALESCE((SELECT MIN(h.n_days) 
     FROM (SELECT unnest(history) as h) 
     WHERE h.template = selected_template), 30) AS days_since_last_sent
FROM sample_data
""")

#show the first 5 rows of the combined view
history_sample = con.execute("""
    SELECT * 
    FROM history_extracted
    LIMIT 5
""").df()
display(history_sample)

,datetime,ui_language,eligible_templates,history,selected_template,session_end_completed,days_since_last_sent
0,6.969815,es,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'G', 'n_days': 29.03557586669922...",L,0,30.000000
1,14.945741,fr,"[K, H, G, E, B, J, L, F, D, A]",[],H,0,30.000000
2,1.607130,en,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'C', 'n_days': 29.22372627258300...",B,0,26.163692
3,9.938345,es,"[K, H, G, E, B, J, L, F, D]","[{'template': 'A', 'n_days': 19.07602691650390...",L,0,2.999995
4,0.458646,de,"[G, E, B, K, H, J, L, F, D]","[{'template': 'B', 'n_days': 4.99090576171875}...",B,0,1.999997


In [30]:
#cut intervals for 'days_since_last_sent' variable
con.execute("""
    CREATE OR REPLACE VIEW history_extracted_intervals AS 
    SELECT 
    *,
    CASE 
        WHEN days_since_last_sent < 3 THEN '0-3 days'
        WHEN days_since_last_sent < 7 THEN '3-7 days'
        WHEN days_since_last_sent < 14 THEN '7-14 days'
        WHEN days_since_last_sent <= 30 THEN '15-30 days'
    END AS days_interval
FROM history_extracted
""")

#calculate the success rate by 'days_interval'
days_success_rates = con.execute("""
    SELECT
        days_interval,
        AVG(session_end_completed) AS success_rate
    FROM history_extracted_intervals
    GROUP BY days_interval
    ORDER BY success_rate DESC
""").df()

fig_days = px.bar(
    days_success_rates, 
    x='days_interval', 
    y='success_rate',
    hover_data={'success_rate': ':.2%'},
    text_auto='.2%', 
    title='Success Rate by Days Since Last Sent',
    labels={'success_rate': 'Success Rate', 'days_interval': 'Days Since Last Sent'},
    color='success_rate',
    color_continuous_scale='Viridis'
)
fig_days.show()